In [ ]:
import sys, os, glob, shutil, time, json, gc
import numpy as np, pandas as pd, torch
t0 = time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t0:7.0f}с] {m}", flush=True)
name = torch.cuda.get_device_name(0); log(f"GPU: {name}")
if "T4" not in name and "L4" not in name and "A100" not in name:
    raise SystemExit(f"нужна T4, выдали {name}")
code = os.path.dirname(glob.glob("/kaggle/input/**/cross_encoder.py", recursive=True)[0])
os.makedirs("/kaggle/working/src", exist_ok=True)
for p in glob.glob(code + "/*.py"): shutil.copy(p, "/kaggle/working/src/")
open("/kaggle/working/src/__init__.py", "a").close()
os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")
from src.cross_encoder import build_product_texts
from transformers import AutoModelForSequenceClassification, AutoTokenizer

def unpack(prefix, tag, where):
    """Веса лежат плоско с префиксом; `from_pretrained` требует каталог."""
    root = os.path.dirname(glob.glob(f"/kaggle/input/**/{prefix}__model.safetensors", recursive=True)[0])
    os.makedirs(where, exist_ok=True)
    for f in ("model.safetensors", "config.json", "tokenizer.json",
              "tokenizer_config.json", "inference_config.json"):
        dst = f"{where}/{f}"
        if not os.path.exists(dst): os.symlink(f"{root}/{prefix}__{f}", dst)
    return where

def score(path, left, right, order, batch=256):
    tok = AutoTokenizer.from_pretrained(path, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        path, local_files_only=True, dtype=torch.float16).cuda().eval()
    out = np.empty(len(left), dtype=np.float32)
    with torch.inference_mode():
        for start in range(0, len(order), batch):
            rows = order[start:start+batch]
            enc = tok(left[rows].tolist(), right[rows].tolist(), padding=True, truncation=True,
                      max_length=256, pad_to_multiple_of=8, return_tensors="pt").to("cuda")
            out[rows] = model(**enc).logits.squeeze(-1).float().cpu().numpy()
    del model; torch.cuda.empty_cache(); gc.collect()
    return out

pack = os.path.dirname(glob.glob("/kaggle/input/**/llm_train.parquet", recursive=True)[0])
big = os.path.dirname(glob.glob("/kaggle/input/**/llm_pairs_2m.parquet", recursive=True)[0])
seen = pd.read_parquet(pack + "/llm_train.parquet", columns=["id1", "id2"])
pairs = pd.read_parquet(big + "/llm_pairs_2m.parquet")
items = pd.read_parquet(big + "/llm_items_2m.parquet")
key = lambda d: pd.MultiIndex.from_arrays([d.id1.to_numpy(), d.id2.to_numpy()])
clean = pairs[~key(pairs).isin(key(seen))].reset_index(drop=True)
# Тот же отбор, что в первом проходе: то же зерно, тот же размер, значит те же пары.
if len(clean) > 400_000:
    clean = clean.sample(400_000, random_state=2026).reset_index(drop=True)
used = pd.unique(np.concatenate([clean.id1.to_numpy(), clean.id2.to_numpy()]))
items = items[items.id.isin(used)].reset_index(drop=True)
log(f"пар {len(clean):,}, карточек {len(items):,}")
texts = build_product_texts(items, "compact")
left = clean.id1.map(texts).fillna("").astype(str).to_numpy()
right = clean.id2.map(texts).fillna("").astype(str).to_numpy()
order = np.argsort(np.fromiter((len(a)+len(b) for a, b in zip(left, right)), dtype=np.int32, count=len(left)))
log("тексты готовы")

# --- Проверка fp16 против fp32 на первых 20 тысячах пар ---
probe = order[:20000]
p32 = score(unpack("ce_relaxed", "ce_relaxed", "/kaggle/working/f32"),
            left, right, probe)
p16 = score(unpack("ce_relaxed16", "ce_relaxed", "/kaggle/working/f16"),
            left, right, probe)
a, b = p32[probe], p16[probe]
log(f"fp32 против fp16: одинаковых {np.mean(a == b):.4%}, макс. расхождение {np.max(np.abs(a-b)):.2e}, "
    f"корреляция {np.corrcoef(a, b)[0,1]:.8f}")

for tag, prefix in (("ce_e5", "ce_e516"), ("ce_ru", "ce_ru16")):
    t = time.perf_counter()
    s = score(unpack(prefix, tag, f"/kaggle/working/{tag}"), left, right, order)
    np.save(f"/kaggle/working/{tag}.npy", s)
    log(f"  {tag}: {time.perf_counter()-t:.0f}с ({len(clean)/(time.perf_counter()-t):.0f} пар/с)")
clean.to_parquet("/kaggle/working/clean_pairs_check.parquet", index=False)
log("готово")
